# monty_demo — Reasoning Layer for Physical-AI Teleop Data

**Pitch artifact for [usemonty.com](https://usemonty.com)** — *real-world data infrastructure for physical AI*.

> _"Teach physical AI from human motion — the trajectories AND the world knowledge the human was implicitly using. Past teleop becomes data + priors; every new task starts with both, and the system gets sharper with every attempt."_

This notebook walks the full pipeline end-to-end on real LeRobot ALOHA data. Three prior coffee-brewing episodes seed the substrate; three subsequent "new attempts" — cross-skill, cross-embodiment, and an outlier — exercise the self-improvement loop. The merged human-priors / data-driven impedance recommendation appears at the end.


In [1]:
import numpy as np

from monty_demo import (
    Episode,
    KnowledgeGraph,
    ingest,
    reason,
    print_brief_diff,
)
from monty_demo._timing import format_timing_table, reset_timings
from monty_demo.schemas import EpisodeSource

reset_timings()
kg = KnowledgeGraph()

## 1. Seed the substrate — three prior brew-coffee episodes

Each `ingest()` runs the full pipeline: load → estimate `k_hat` → segment phases → infer intent / objects / skills (from `REPO_METADATA`) → outlier check → write to KG.

In [2]:
for i in range(3):
    ep = Episode.from_lerobot('lerobot/aloha_static_coffee', index=i)
    populated = ingest(kg, ep)
    print(f'  ingested ep {i}: {populated.n_frames:>4} frames @ {populated.source.fps} fps   '
          f'phases={len(populated.phases or [])}   intent={populated.intent.name}')

print()
print('KG state:', kg.stats())

  ingested ep 0: 1100 frames @ 50.0 fps   phases=22   intent=brew-coffee
  ingested ep 1: 1100 frames @ 50.0 fps   phases=27   intent=brew-coffee
  ingested ep 2: 1100 frames @ 50.0 fps   phases=23   intent=brew-coffee

KG state: {'nodes': {'Episode': 3, 'Embodiment': 1, 'Intent': 1, 'Phase': 72, 'Skill': 3, 'Object': 3, 'SafetyTag': 3}, 'edges': {'ON_EMBODIMENT': 3, 'HAS_INTENT': 3, 'HAS_PHASE': 72, 'USES_SKILL': 9, 'INVOLVES': 9, 'HAS_SAFETY_CONTEXT': 3}, 'total_nodes': 86, 'total_edges': 99}


## 2. Reason about a new task (BEFORE any new data)

Target: brew coffee on the same ALOHA bimanual rig, with `mug` as the target object. The brief should be supported by 3 matched episodes, with the human prior on `mug` (`fragility=moderate`, `safety_context=["contains_liquid"]`, `suggested_impedance="gentle"`) already merged into the recommended impedance regime.

In [3]:
brief_before = reason(
    kg,
    intent='brew-coffee',
    target_objects=('mug', 'coffee-machine'),
    embodiment='aloha-bimanual',
)

print('matched_episodes:           ', brief_before.matched_episodes)
print('confidence:                 ', brief_before.confidence)
print('embodiment_diversity:       ', brief_before.embodiment_diversity)
print('recommended_impedance:      ', brief_before.recommended_impedance_regime)
print('safety_warnings:            ', brief_before.safety_warnings)
print('suggested_skills:           ', brief_before.suggested_skills)
print()
print('plan:')
for p in brief_before.plan:
    print(f'  {p.name:<10} ~{p.expected_duration_s:.2f}s   k_hat band {p.suggested_k_hat_band}   '
          f'(supported by {p.n_supporting_episodes} episodes)')
print()
print('notes:')
for n in brief_before.notes:
    print('  -', n)

matched_episodes:            ['ep:lerobot/aloha_static_coffee/0', 'ep:lerobot/aloha_static_coffee/1', 'ep:lerobot/aloha_static_coffee/2']
confidence:                  1.0
embodiment_diversity:        1
recommended_impedance:       gentle
safety_warnings:             ['contains_liquid → spill risk during contact phase', 'electrical → de-energize before contact', 'hot_surface → thermal-rated end-effector required']
suggested_skills:            ['fine-bimanual-coordination', 'place', 'press-button']

plan:
  contact    ~1.94s   k_hat band (0.0, 0.135)   (supported by 3 episodes)
  manipulate ~1.56s   k_hat band (0.0, 0.381)   (supported by 3 episodes)
  retract    ~1.96s   k_hat band (0.0, 0.0)   (supported by 2 episodes)

notes:
  - data-driven k_hat (0.0, 0.14) conflicts with priors on ['coffee-machine'] — defaulting to most cautious regime 'gentle'


## 3. Self-improvement loop

**Reading guide.** Each of the next three cells **ingests one new episode** and then **re-asks the same question**: *what should I do for `brew-coffee` on `aloha-bimanual`?* The query is fixed; the evidence in the KG evolves. Watch the brief change.

Why fix the query? Because the interesting capabilities aren't *retrieval* (which is trivial when you ask about exactly what you just ingested). They're how **non-matching, weakly-matching, and anomalous data** still meaningfully shape the brief for the *original* task:

1. **Cross-skill transfer** — ingest a battery-insertion episode (different intent, shared dexterity skill). Watch `transferable_skills_observed` populate; `matched_episodes` should *not* increase.
2. **Cross-embodiment calibration** — ingest a Koch single-arm episode (hand-labeled `brew-coffee`, different embodiment). Watch `confidence` *drop* despite `matched_episodes` rising.
3. **Outlier detection** — ingest a deliberately anomalous attempt (extended-stall contact phase). Watch `outlier_phases` fire with z-scores ≥ 2.

Together: transfer + calibration + critical sense.

### Attempt 1 — cross-skill (battery insertion)

Different intent (`insert-battery`), different objects, but **shares `fine-bimanual-coordination`** with brew-coffee. Crucially, it brings in two new skills the coffee episodes don't have: **`precision-insert`** and **`align-and-press`**.

The brew-coffee `suggested_skills` already includes `insert-pod` (the filter-pod handling step). The reasoner's job here is to surface that the new `precision-insert` + `align-and-press` skills observed in the battery task are **candidate refinements to that existing `insert-pod` step**.

**Watch for:** `matched_episodes` stays at 3 (battery doesn't match the brew-coffee intent). `new transferable skills` lists `precision-insert` and `align-and-press` — surfaced via the shared `fine-bimanual-coordination` adjacency. The reasoner is saying *"your insert-pod step in brew-coffee could be informed by the precision-insert / align-and-press primitives we just observed in battery handling."*

In [ ]:
ep = Episode.from_lerobot('lerobot/aloha_static_battery', index=0)
ingest(kg, ep)

brief_after_1 = reason(kg, intent='brew-coffee', target_objects=('mug', 'coffee-machine'), embodiment='aloha-bimanual')
print('--- diff after Attempt 1 ---')
print_brief_diff(brief_before, brief_after_1, kg)

### Attempt 2 — cross-embodiment (Koch single-arm)

Same intent (hand-labeled `brew-coffee` with `source=manual, confidence=0.4`), but a single-arm Koch instead of bimanual ALOHA. The reasoner's `*0.6` score penalty on cross-embodiment + `*0.85` confidence dip combine.

**Watch for:** `matched_episodes` rises (Koch matches by intent), but `confidence` *drops*. That's the calibration story — adding less-reliable data should *reduce* certainty, not boost it. `embodiment_diversity` flips from 1 to 2.

In [5]:

ep = Episode.from_lerobot('lerobot/koch_pick_place_5_lego', index=0)
ingest(kg, ep)

brief_after_2 = reason(kg, intent='brew-coffee', target_objects=('mug', 'coffee-machine'), embodiment='aloha-bimanual')
print('--- diff after Attempt 2 ---')
print_brief_diff(brief_after_1, brief_after_2, kg)
print()
print(f'embodiment_diversity went {brief_after_1.embodiment_diversity} → {brief_after_2.embodiment_diversity}')

--- diff after Attempt 2 ---
matched_episodes:        3 → 4
contact  stiffness band:  (0.0, 0.135) → (0.0, 0.095)   [tightened]
manipulate stiffness band:  (0.0, 0.381) → (0.0, 0.243)   [tightened]
confidence:              1.0 → 0.85   (↓ -0.150)
new skills:              ['pick']
outlier phase:           manipulate (z=1.63) on ep:lerobot/koch_pick_place_5_lego/0 [info]
outlier phase:           manipulate (z=3.37) on ep:lerobot/koch_pick_place_5_lego/0 [alert]
outlier phase:           manipulate (z=2.94) on ep:lerobot/koch_pick_place_5_lego/0 [warning]
outlier phase:           manipulate (z=4.06) on ep:lerobot/koch_pick_place_5_lego/0 [alert]
outlier phase:           manipulate (z=4.8) on ep:lerobot/koch_pick_place_5_lego/0 [alert]
outlier phase:           manipulate (z=3.74) on ep:lerobot/koch_pick_place_5_lego/0 [alert]

embodiment_diversity went 1 → 2


### Attempt 3 — outlier detection

An attempt whose contact phase is dramatically longer than the established baseline. In production this would be a fumble or a partial failure flagged for review; here we synthesize one by extending a coffee episode's mid-task with a 200-frame stall, framed honestly as a deterministic stand-in.

**Watch for:** `outlier_phases` fires with z-scores ≥ 2 against same-named phases already in the KG. The bad-data flag appears *before* the new episode is allowed to influence future briefs — i.e., the system has critical sense about its own input quality.

In [6]:
# Honest framing: in production this is a real attempt that segments anomalously.
# Here we synthesize one to demonstrate the capability deterministically.
real_ep = Episode.from_lerobot('lerobot/aloha_static_coffee', index=3)
T = real_ep.n_frames
# Extend by holding the joints frozen for 200 extra frames mid-episode (simulates a stall)
EXTRA = 200
stall_at = T // 2
extended_pos = np.concatenate([
    real_ep.joint_positions[:stall_at],
    np.repeat(real_ep.joint_positions[stall_at:stall_at+1], EXTRA, axis=0),
    real_ep.joint_positions[stall_at:],
], axis=0).astype(np.float32)
extended_act = np.concatenate([
    real_ep.joint_actions[:stall_at],
    np.linspace(real_ep.joint_actions[stall_at], real_ep.joint_actions[stall_at]+0.5, EXTRA).astype(np.float32),
    real_ep.joint_actions[stall_at:],
], axis=0).astype(np.float32)
v = np.linalg.norm(np.diff(extended_pos, axis=0), axis=1) / real_ep.dt
ee_v = np.concatenate([[v[0]], v]).astype(np.float32)
outlier_ep = Episode(
    source=EpisodeSource(repo_id='lerobot/aloha_static_coffee', index=999, embodiment='aloha-bimanual', fps=real_ep.source.fps),
    n_frames=extended_pos.shape[0], dt=real_ep.dt,
    joint_positions=extended_pos, joint_actions=extended_act,
    ee_velocity_norm=ee_v,
)
ingest(kg, outlier_ep)

brief_final = reason(kg, intent='brew-coffee', target_objects=('mug', 'coffee-machine'), embodiment='aloha-bimanual')
print('--- diff after Attempt 3 (outlier) ---')
print_brief_diff(brief_after_2, brief_final, kg)

--- diff after Attempt 3 (outlier) ---
matched_episodes:        4 → 5
confidence:              0.85 → 0.85   (· +0.000)
outlier phase:           approach (z=36.35) on ep:lerobot/aloha_static_coffee/999 [alert]
outlier phase:           contact (z=2.28) on ep:lerobot/aloha_static_coffee/999 [warning]
outlier phase:           retract (z=9.16) on ep:lerobot/aloha_static_coffee/999 [alert]


'matched_episodes:        4 → 5\nconfidence:              0.85 → 0.85   (· +0.000)\noutlier phase:           approach (z=36.35) on ep:lerobot/aloha_static_coffee/999 [alert]\noutlier phase:           contact (z=2.28) on ep:lerobot/aloha_static_coffee/999 [warning]\noutlier phase:           retract (z=9.16) on ep:lerobot/aloha_static_coffee/999 [alert]'

## 4. Cumulative diff — start to finish

What did the system *learn* across the three new attempts?

In [7]:
print('=== brief_before vs brief_final ===')
print_brief_diff(brief_before, brief_final)
print()
print('confidence:', brief_before.confidence, '→', brief_final.confidence)
print('matched:   ', len(brief_before.matched_episodes), '→', len(brief_final.matched_episodes))
print('embodiments:', brief_before.embodiment_diversity, '→', brief_final.embodiment_diversity)

=== brief_before vs brief_final ===
matched_episodes:        3 → 5
manipulate stiffness band:  (0.0, 0.381) → (0.0, 0.243)   [tightened]
confidence:              1.0 → 0.85   (↓ -0.150)
new transferable skills: ['pinch-grasp', 'thread']
new skills:              ['pick']

confidence: 1.0 → 0.85
matched:    3 → 5
embodiments: 1 → 2


## 5. Human priors in action

The brief surfaces both the data-driven `k_hat` band (from contact-phase aggregation across matched episodes) and the **merged** `recommended_impedance_regime` after applying the human's prior knowledge of the target objects. The data alone might say "around 0.3–0.6"; the prior on `mug` (`fragility=moderate`, `contains_liquid`, `suggested_impedance=gentle`) tightens it down.

In [8]:
print('object_knowledge:')
for ok in brief_final.object_knowledge:
    print(f'  {ok.name:<16} fragility={ok.fragility:<10} mass={ok.mass_category:<6} '
          f'safety={ok.safety_context}  prior_impedance={ok.suggested_impedance}')
print()
contact_band = next((p.suggested_k_hat_band for p in brief_final.plan if p.name == 'contact'), None)
print(f'data-driven contact k_hat band:    {contact_band}')
print(f'recommended_impedance_regime:     {brief_final.recommended_impedance_regime}')
print()
print('safety_warnings:')
for w in brief_final.safety_warnings:
    print('  -', w)

object_knowledge:
  mug              fragility=moderate   mass=light  safety=['contains_liquid']  prior_impedance=gentle
  coffee-machine   fragility=robust     mass=heavy  safety=['hot_surface', 'electrical']  prior_impedance=firm

data-driven contact k_hat band:    (0.0, 0.213)
recommended_impedance_regime:     gentle

safety_warnings:
  - contains_liquid → spill risk during contact phase
  - electrical → de-energize before contact
  - hot_surface → thermal-rated end-effector required


## 6. Cypher export — the migration path

NetworkX is the in-memory backend; `kg.to_cypher()` emits the equivalent Cypher for the same query, signalling the path to a real graph DB (Kuzu / Neo4j) at scale.

In [9]:
print(kg.to_cypher({'intent': 'brew-coffee', 'embodiment': 'aloha-bimanual', 'phase': 'contact', 'min_k_hat': 0.2}))

MATCH (e:Episode), (e)-[:HAS_INTENT]->(i:Intent {name: 'brew-coffee'}), (e)-[:ON_EMBODIMENT]->(em:Embodiment {name: 'aloha-bimanual'}), (e)-[:HAS_PHASE]->(p:Phase {name: 'contact'})
WHERE p.k_hi >= 0.2
RETURN e


## 7. Timing — speed as evidence

Every public op self-times. This run's actuals:

In [10]:
print(format_timing_table())

name                                 calls  total_ms  mean_ms  median_ms  p95_ms 
-----------------------------------  -----  --------  -------  ---------  -------
monty_demo.Episode.from_lerobot      6      1630.667  271.778  229.004    509.472
monty_demo.KnowledgeGraph.add        6      1.047     0.175    0.171      0.27   
monty_demo._io.load_lerobot_episode  6      1629.906  271.651  228.867    509.309
monty_demo.estimate_stiffness        6      1.827     0.304    0.317      0.368  
monty_demo.ingest                    6      12.841    2.14     2.266      3.33   
monty_demo.reason                    4      3.386     0.847    0.818      1.12   
monty_demo.segment_phases            6      0.769     0.128    0.135      0.177  


## What this enables

- **Every new task starts with retrieved priors** instead of cold — phase plan, stiffness band, suggested skills, object-aware safety.
- **Cross-task transfer is structural, not just label-based** — `transferable_skills_observed` surfaces dexterity learned in adjacent tasks.
- **Bad new data is flagged before it pollutes downstream training** — z-score outlier detection on every ingest.
- **Human knowledge of the world is first-class** — fragility, mass class, safety context, suggested impedance, all merged into the brief alongside the data-driven signal.
- **Confidence is calibrated** — adding cross-embodiment evidence *lowers* confidence, not raises it. A reasoner that blindly grows confidence with more data is broken.
